<img src='https://repository-images.githubusercontent.com/121802384/c355bb80-7d42-11e9-9e0e-4729609f9fbc' alt='WRF-Hydro Logo' width="15%"/>

# Lesson 6 - Lake options for the NWM configuration

## Overview
In [Lesson 4](Lesson-4-run-options.ipynb) we covered some experiments in tinkering with terrain physics. In this lesson, we specifically experiment with lake physics options. 

We conduct two experiments in this lesson: 

1. Removing lakes from the simulation
2. Modifying lake parameters

See [here](https://ral.ucar.edu/sites/default/files/public/Channel%20Routing_Lakes.pdf) for reference material on lakes and channel routing in WRF-Hydro.

### Compiling WRF-Hydro
We do not need to modify the code binary to conduct lake physics experiments, so we will use the compiled code from the template directory.  

If you need to recompile the code, refer to the instructions in [Lesson 2](Lesson-2-run.ipynb).

### Creating the simulation directory and running a baseline WRF-Hydro simulation
Below are the commands from Lesson 2 to create the simulation directory and run a WRF-Hydro simulation. This simulation will be run exactly as was done in Lesson 2 and will serve as our baseline simulation. We will modify run-time options in subsequent simulations in the lesson and compare the outputs.  

In [ ]:
%%bash

# Make a new directory for our baseline simulation
mkdir -p ~/wrf_hydro_training/output/lesson6/run_nwm_baseline

# Copy our model files to the simulation directory
cp ~/wrf_hydro_training/wrf_hydro_nwm_public/build/Run/*.TBL \
  ~/wrf_hydro_training/output/lesson6/run_nwm_baseline
cp ~/wrf_hydro_training/wrf_hydro_nwm_public/build/Run/wrf_hydro \
  ~/wrf_hydro_training/output/lesson6/run_nwm_baseline

# Create symbolic links to large domain files
cp -as $HOME/wrf_hydro_training/example_case/FORCING \
  ~/wrf_hydro_training/output/lesson6/run_nwm_baseline
cp -as $HOME/wrf_hydro_training/example_case/NWM/DOMAIN \
  ~/wrf_hydro_training/output/lesson6/run_nwm_baseline
cp -as $HOME/wrf_hydro_training/example_case/NWM/RESTART \
  ~/wrf_hydro_training/output/lesson6/run_nwm_baseline

# Copy namelist files
cp ~/wrf_hydro_training/example_case/NWM/namelist.hrldas \
  ~/wrf_hydro_training/output/lesson6/run_nwm_baseline
cp ~/wrf_hydro_training/example_case/NWM/hydro.namelist \
  ~/wrf_hydro_training/output/lesson6/run_nwm_baseline

**Run the simulation**

In [ ]:
%%bash
cd ~/wrf_hydro_training/output/lesson6/run_nwm_baseline
mpirun -np 2 ./wrf_hydro &> run.log

In [ ]:
%%bash
cd ~/wrf_hydro_training/output/lesson6/run_nwm_baseline

In [ ]:
%%bash

# Check that it finished successfully
tail -1 ~/wrf_hydro_training/output/lesson6/run_nwm_baseline/diag_hydro.00000

## Setup for experiments

We will create a new simulation directory for each experiment. First, we will make a new simulation directory as we did with the baseline run above and use this as a template for creating multiple new simulation directories.

**Step 1: Create a simulation directory to use as a template**

In [ ]:
%%bash

# Make a new directory for our baseline simulation
rm -rf ~/wrf_hydro_training/output/lesson6/run_nwm_template
mkdir -p ~/wrf_hydro_training/output/lesson6/run_nwm_template

# Copy our model files to the simulation directory
cp ~/wrf_hydro_training/wrf_hydro_nwm_public/build/Run/*.TBL \
  ~/wrf_hydro_training/output/lesson6/run_nwm_template
cp ~/wrf_hydro_training/wrf_hydro_nwm_public/build/Run/wrf_hydro \
  ~/wrf_hydro_training/output/lesson6/run_nwm_template

# Copy the domain files so we can modify them
cp -r ~/wrf_hydro_training/example_case/NWM/DOMAIN \
  ~/wrf_hydro_training/output/lesson6/run_nwm_template

# Create symbolic links to large domain files
cp -as $HOME/wrf_hydro_training/example_case/FORCING \
  ~/wrf_hydro_training/output/lesson6/run_nwm_template
cp -as $HOME/wrf_hydro_training/example_case/NWM/RESTART \
  ~/wrf_hydro_training/output/lesson6/run_nwm_template

# Copy namelist files
cp ~/wrf_hydro_training/example_case/NWM/namelist.hrldas \
  ~/wrf_hydro_training/output/lesson6/run_nwm_template
cp ~/wrf_hydro_training/example_case/NWM/hydro.namelist \
  ~/wrf_hydro_training/output/lesson6/run_nwm_template

**Step 2: View the contents**

In [ ]:
%%bash

ls ~/wrf_hydro_training/output/lesson6/run_nwm_template

### Review the baseline
First, we will recall the hydrograph from location below the lake, at the outlet of the basin: 

In [ ]:
# Load the required libraries
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd

xr.set_options(display_style="html")
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

In [ ]:
chanobs_baseline = xr.open_mfdataset('~/wrf_hydro_training/output/lesson6/run_nwm_baseline/*CHANOBS*',
                            combine='by_coords')
obs = pd.read_csv('~/wrf_hydro_training/example_case/USGS_obs.csv',dtype=str)
obs['dateTime'] = pd.to_datetime(obs['dateTime'])
obs['streamflow_cms'] = pd.to_numeric(obs['streamflow_cms'])

**Plot the hydrograph for the gage below the reservoir (feature_id = 6191738, USGS ID 01363556)**

Table 1. USGS Stream Gage IDs and associated WRF-Hydro feature_id indices.

|USGS Gage ID|WRF-Hydro gage feature_id|Gage Information|
|------|:-:|---|
| 01362200 | 6189624 |https://waterdata.usgs.gov/pa/nwis/uv/?site_no=01362200|
| 01362500 | 6189754 |https://waterdata.usgs.gov/nwis/nwismap/?site_no=01362500|
| 01363556 | 6191738 |https://waterdata.usgs.gov/nwis/nwismap/?site_no=01363556|

In [ ]:
# Specify our evaluation gage and associated reach ID
gageid = "01363556"
fid = 6191738

**Plot the default hydrograph at the stream location downstream of the lake**

In [ ]:
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

fig, axes = plt.subplots(ncols=1,figsize=(12, 6))
plt.suptitle('Hydrographs for lakes on and off at stream location downstream of lake',fontsize=24)
chanobs_baseline.sel(feature_id = fid).streamflow.plot(label='Lakes on',
                                                        color='black',
                                                        linestyle='--')
obs[obs['site_no'] == gageid].plot(x='dateTime',
                                       y='streamflow_cms',
                                       ax=axes,
                                       label='Observed',
                                       color='grey')
plt.legend()
plt.show()

From this default simulation, we have some work to do for improving the lake's representation.  First, let's see what happens when the lake is removed completely.

## Experiment 1 - Removing lakes from the simulation

### Background
In the NWM configuration, lakes are represented as objects connected by channel reaches, rather than explicitly defined on the grid (as they are in the gridded channel configurations). In reach-based configurations like the NWM, network topology is maintained with or without the lakes, so we can turn them on and off with a namelist change.  

The only changes to execute with and without lakes are edits in the `hydro.namelist`. See below for details.

### Objective
Run a simulation for the NWM configuration without representing lakes and compare the resulting hydrographs at a gage downstream of the lake. 

### Create experiment simulation directory
First we will copy our template directory `~/wrf_hydro_training/output/lesson6/run_nwm_template` to create a new simulation directory for this experiment.

In [ ]:
%%bash
cp -r ~/wrf_hydro_training/output/lesson6/run_nwm_template \
~/wrf_hydro_training/output/lesson6/run_lakes_off

### Change the namelist setting and run the simulation

**Step 1: Edit the hydro.namelist to deactivate the level-pool reservoir scheme and instead turn lakes completely off and route water through internal lake reaches:**

There are two options in the [hydro.namelist](hydro.namelist) to turn lakes off.

First is changing the lake physics option from level-pool (option 1) to off (option 0). 

```
! Lake / Reservoir options (0=lakes off, 1=level pool (typical default),
!                           2=passthrough, 3=reservoir DA [see &reservoir_nlist below])
lake_option = 0
```

Second is turning off the lakeout output files.

```
outlake  = 0                 ! Netcdf grid of lake values (1d)
                             !      0 = no output, 1 = output
``` 

**Step 2: Run the simulation**

In [ ]:
%%bash

cd ~/wrf_hydro_training/output/lesson6/run_lakes_off
mpirun -np 2 ./wrf_hydro &> run.log

**Step 3: Check that the simulation finished successfully**

In [ ]:
%%bash

tail -1 ~/wrf_hydro_training/output/lesson6/run_lakes_off/diag_hydro.00000

### Results
We will now look at the differences in streamflow between our baseline run with lakes on and our experiment with lakes off.

We will use Python and the `xarray` library to load the data and plot hydrographs. For an intro to these tools, please see [Lesson 3](Lesson-3-visualize.ipynb).

**Load the CHANOBS streamflow datasets**

We are going to use the CHANOBS files because it will limit the number of reaches to only those which we have specified have a gage.

In [ ]:
chanobs_lakes_off = xr.open_mfdataset('~/wrf_hydro_training/output/lesson6/run_lakes_off/*CHANOBS*',
                            combine='by_coords')

**Plot the hydrographs at the stream location downstream of the lake**

In [ ]:
fig, axes = plt.subplots(ncols=1,figsize=(12, 6))
plt.suptitle('Hydrographs for lakes on and off at stream location downstream of lake',fontsize=24)
chanobs_lakes_off.sel(feature_id = fid).streamflow.plot(label='Lakes off',
                                                        color='blue',
                                                        linestyle='-')
chanobs_baseline.sel(feature_id = fid).streamflow.plot(label='Lakes on',
                                                        color='black',
                                                        linestyle='--')
obs[obs['site_no'] == gageid].plot(x='dateTime',
                                       y='streamflow_cms',
                                       ax=axes,
                                       label='Observed',
                                       color='grey')
plt.ylim(0,300)
plt.legend()
plt.show()

### Discussion
At the gage location downstream of the lake we see that streamflow is considerably higher in the simulation without lakes represented. This reflects the expected behavior of the lake attenuating streamflow at the outlet due to storage. 

### Summary
The 'lakes-off' simulation shows a higher peak compared with the 'lakes-on' and observed hydrographs. We confirm that this lake serves to attenuate flow during a high event.

# Experiment 2 - Modifying lake parameters in `LAKEPARM.nc`

### Background
The orifice and weir lake parameters populated into LAKEPARM.nc by the GIS pre-processing tools are essentially placeholders for the user to replace with their own data. The accuracy of these parameters can have a significant impact on the storage and thus outflow of the lake. **We highly recommend replacing the default lake parameters with known values!**

### Objective
In this exercise, we will update one of these parameters, the weir length - `WeirL`, with an estimate from observations and explore the impact on the streamflow at the downstream gauge.

### First, let's make a new directory to conduct this experiment.

In [ ]:
%%bash
rm -r ~/wrf_hydro_training/output/lesson6/run_lakeparm_edit
cp -r ~/wrf_hydro_training/output/lesson6/run_nwm_template \
  ~/wrf_hydro_training/output/lesson6/run_lakeparm_edit

### Before modifying anything, let's check the current default parameters in `LAKEPARM.nc`

In [ ]:
%%bash

ncdump ~/wrf_hydro_training/output/lesson6/run_lakeparm_edit/DOMAIN/LAKEPARM.nc | tail -35

### Edit the lake parameter file and run the simulation
**Step 1: Edit the `LAKEPARM.nc` file**

We will increase the maximum lake elevation while maintaining the relative weir capacity, as well as increase the weir coefficient. We will also remove the orifice by setting its area to 0.

NCO (NetCDF Operators, http://nco.sourceforge.net/) is a set of useful utilities to manipulate NetCDF files. We will use the NCO command ncap2 to edit the WeirL variable.

In [ ]:
%%bash

ncap2 -O -s "LkMxE=190" ~/wrf_hydro_training/output/lesson6/run_lakeparm_edit/DOMAIN/LAKEPARM.nc \
~/wrf_hydro_training/output/lesson6/run_lakeparm_edit/DOMAIN/LAKEPARM_mods.nc

ncap2 -O -s "WeirE=187" ~/wrf_hydro_training/output/lesson6/run_lakeparm_edit/DOMAIN/LAKEPARM_mods.nc \
~/wrf_hydro_training/output/lesson6/run_lakeparm_edit/DOMAIN/LAKEPARM_mods.nc

ncap2 -O -s "WeirC=0.9" ~/wrf_hydro_training/output/lesson6/run_lakeparm_edit/DOMAIN/LAKEPARM_mods.nc \
~/wrf_hydro_training/output/lesson6/run_lakeparm_edit/DOMAIN/LAKEPARM_mods.nc

ncap2 -O -s "OrificeA=0" ~/wrf_hydro_training/output/lesson6/run_lakeparm_edit/DOMAIN/LAKEPARM_mods.nc \
~/wrf_hydro_training/output/lesson6/run_lakeparm_edit/DOMAIN/LAKEPARM_mods.nc

**Step 2: Verify that the change was made**

We will check the value of the parameters in our new `LAKEPARM_mods.nc` file against the original `LAKEPARM.nc` file.

In [ ]:
%%bash

ncdump ~/wrf_hydro_training/output/lesson6/run_lakeparm_edit/DOMAIN/LAKEPARM_mods.nc | tail -35

**Step 3: Edit the *hydro.namelist* to use the new *LAKEPARM_mods.nc***

We will change the `route_lake_f` option in the [hydro.namelist](~/wrf_hydro_training/output/lesson6/run_lakeparm_edit/hydro.namelist) to use the edited lakeparm file. 

```
! Specify the lake parameter file (e.g.: "LAKEPARM.nc").
! Note REQUIRED if lakes are on.
route_lake_f = './DOMAIN/LAKEPARM_mods.nc'
```

**Step 4: Run the simulation**

In [ ]:
%%bash

cd ~/wrf_hydro_training/output/lesson6/run_lakeparm_edit
mpirun -np 2 ./wrf_hydro &> run.log

In [ ]:
%%bash

tail -1 ~/wrf_hydro_training/output/lesson6/run_lakeparm_edit/diag_hydro.00000

### Results
We will now look at the differences in streamflow between our baseline run with the original WeirL of 10m and our edited WeirL of 677m.

We will use Python and the `xarray` library to load the data and plot hydrographs. For an intro to these tools, please see [Lesson 3](Lesson-3-visualize.ipynb).

**Load the CHANOBS streamflow datasets**

We are going to use the *CHANOBS* files because it will limit the number of grid cells to only those which we have specified have a gage.

In [ ]:
chanobs_lakeparm_edit = xr.open_mfdataset('~/wrf_hydro_training/output/lesson6/run_lakeparm_edit/*CHANOBS*',
                            combine='by_coords')

**Plot the hydrographs**

In [ ]:
fig, axes = plt.subplots(ncols=1,figsize=(12, 6))
plt.suptitle('Hydrograph below lake for baseline and modified parameters',fontsize=24)
chanobs_lakeparm_edit.sel(feature_id = fid).streamflow.plot(label='Lake Mods',
                                                        color='blue',
                                                        linestyle='-')
chanobs_baseline.sel(feature_id = fid).streamflow.plot(label='Baseline',
                                                        color='black',
                                                        linestyle='--')
obs[obs['site_no'] == gageid].plot(x='dateTime',
                                       y='streamflow_cms',
                                       ax=axes,
                                       label='Observed',
                                       color='grey')
plt.ylim(0,120)
plt.legend()
plt.show()

**Compare the result from experiment 1, where the lake was removed.**

In [ ]:
fig, axes = plt.subplots(ncols=1,figsize=(12, 6))
plt.suptitle('Hydrographs of baseline, no-lake, and modified weir length.',fontsize=24)
chanobs_lakeparm_edit.sel(feature_id = fid).streamflow.plot(label='Lake Mods',
                                                        color='blue',
                                                        linestyle='-')
chanobs_baseline.sel(feature_id = fid).streamflow.plot(label='Baseline',
                                                        color='black',
                                                        linestyle='--')
chanobs_lakes_off.sel(feature_id = fid).streamflow.plot(label='Lake off',
                                                        color='red',
                                                        linestyle='--')
obs[obs['site_no'] == gageid].plot(x='dateTime',
                                       y='streamflow_cms',
                                       ax=axes,
                                       label='Observed',
                                       color='grey')
plt.ylim(0,300)
plt.legend()
plt.show()

### Summary
The increase in weir length causes more water to discharge from the lake earlier than the default. The difference in initial condition for the modified WeirL expt is due to not spinning up with this parameter, and the lake initially discharged to meet its equilibrium. When an abrupt parameter change is made without proper spinup, these artifacts are possible. 

**Load the LAKEOUT files to investigate three lake-related variables: inflow, outflow, and water surface elevation.**

In [ ]:
lakeout_baseline = xr.open_mfdataset('~/wrf_hydro_training/output/lesson6/run_nwm_baseline/*LAKEOUT*',
                            combine='by_coords')
lakeout_lakeparm_edit = xr.open_mfdataset('~/wrf_hydro_training/output/lesson6/run_lakeparm_edit/*LAKEOUT*',
                            combine='by_coords')

In [ ]:
fig, axes = plt.subplots(ncols=1,figsize=(12, 6))
plt.suptitle('Lake output for Ashokan Reservoir, baseline and modified parameters',fontsize=24)

lakeout_lakeparm_edit.sel(feature_id = 6191414).water_sfc_elev.plot(label='Lake Mods',
                                                        color='blue',
                                                        linestyle='-')
lakeout_baseline.sel(feature_id = 6191414).water_sfc_elev.plot(label='Baseline',
                                                        color='black',
                                                        linestyle='--')
plt.legend()
plt.show()

### **On your own experiments**: 
In the above cells, modify the `inflow` variable selection in the plotting function above to `outflow`, then to `water_sfc_elev`.  Feel free to copy and paste the plotting cells so that each of the plots are saved. 

### Discussion
**Streamflow**: Not surprisingly, extending the maximum lake elevation via the `LkMxE` parameter caused the lake to discharge less water.  Also, closing the orifice drainage reduced the slow lake discharge. Ideally we would update these parameters based on actual measurements and run a longer simulation to equilibriate lake levels before the event.

**Lake outputs**: From the lake level perspective, the baseline simulation shows that the lake fills and then slowly drains, while our modified experiment does not fill until the end of the event.

# Next up - Exploring gridded vs. NWM configurations
This concludes lesson 6. 

In the [next lesson](Lesson-7-configurations.ipynb), we will explore the differences between the gridded and NWM configurations.

**IT IS BEST TO EITHER SHUTDOWN THIS LESSON OR CLOSE IT BEFORE PROCEEDING TO THE NEXT LESSON TO AVOID POSSIBLY EXCEEDING ALLOCATED MEMORY. Shutdown the lesson be either closing the browser tab for the lesson or selecting `Kernel -> Shut Down Kernel` in JupyterLab.**

© UCAR 2025